# Probe: Stage 5 GPU validation on Google Colab

Author: Silas Offei Darko

This notebook runs Probe's hardware validation on the NVIDIA GPU that Colab assigns, and packs all of the evidence into one archive.

1. **Runtime → Change runtime type → GPU** (a T4 is enough), then connect.
2. **Runtime → Run all.**
3. When asked, upload `probe.bundle`.
4. When it finishes, download the `probe-stage5-colab-….tar.gz` that it offers.

The notebook stops at the first stage that cannot continue, and still packs whatever it recorded. It never substitutes CPU-only or synthetic results. The bottom line reads **STAGE 5 HARDWARE VALIDATION FINISHED** only if CUDA workloads actually executed on the GPU.

Expect roughly 30–60 minutes for a full run on a T4. The two clean builds and the compute-sanitizer passes take most of it.

In [ ]:
# Shared helpers. Every later cell uses these; nothing here touches the GPU.
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path(os.environ.get("PROBE_NB_ROOT", "/content"))  # Colab's working directory
REPO = ROOT / "probe"
BUNDLE = ROOT / "probe.bundle"
WORK = ROOT / "probe-stage5"  # everything this notebook records
LOGS = WORK / "logs"
STARTED = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
STATUS = {"started_utc": STARTED, "stages": {}}

# Colab keeps nvcc, cuobjdump and compute-sanitizer in /usr/local/cuda/bin.
CUDA_BIN = Path("/usr/local/cuda/bin")
if CUDA_BIN.is_dir() and str(CUDA_BIN) not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = f"{CUDA_BIN}{os.pathsep}{os.environ['PATH']}"
LOGS.mkdir(parents=True, exist_ok=True)

# Steps of validate_gpu.py whose failure means Probe itself has a problem.
# env/ steps only describe the machine; sanitizer steps are judged separately.
BLOCKING_PREFIXES = (
    "build/",
    "device/",
    "vector_add/",
    "reduction/",
    "timing/",
    "warmup/",
    "noise/",
    "workflow/",
)


# Terminal color codes some tools emit even when piped; stripped from what is
# recorded so logs and summaries are plain text. Nothing else is altered.
ANSI_ESCAPE = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")


class StageStopped(Exception):
    """Raised to stop 'Run all' at the first stage that cannot continue."""


def sh(name, command, cwd=None, quiet=False):
    """Run a shell command, stream its output, save it to logs/<name>.log.
    Returns (exit code, output)."""
    log = LOGS / f"{name}.log"
    print(f"$ {command}")
    proc = subprocess.Popen(
        command,
        shell=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )
    lines = []
    with open(log, "w") as f:
        f.write(f"$ {command}\n")
        for line in proc.stdout:
            line = ANSI_ESCAPE.sub("", line)
            lines.append(line)
            f.write(line)
            if not quiet:
                print(line, end="", flush=True)
        code = proc.wait()
        f.write(f"\n[exit {code}]\n")
    print(f"[exit {code}]")
    return code, "".join(lines)


def save_status():
    (WORK / "status.json").write_text(json.dumps(STATUS, indent=2, default=str) + "\n")


def version_tuple(text):
    return tuple(int(p) for p in re.findall(r"\d+", text)[:3])


def gpu_slug(name):
    return re.sub(r"[^a-z0-9]+", "-", (name or "unknown-gpu").lower()).strip("-") or "unknown-gpu"


def find_run_dir(output):
    """The directory validate_gpu.py reports as 'recorded in <dir>'."""
    found = re.findall(r"^recorded in (\S+)\s*$", output, re.M)
    return Path(found[-1]) if found else None


def _read(base, stream):
    path = Path(f"{base}.{stream}")
    return path.read_text(errors="replace") if path.is_file() else ""


def sanitizer_verdict(step, text):
    """clean / errors reported / unavailable / did not complete. 'clean' needs
    both exit 0 and the sanitizer's own zero-error summary line."""
    if step["note"] and "command not found" in step["note"]:
        return "unavailable"
    if step["exit_code"] == 99 or re.search(r"(ERROR SUMMARY|RACECHECK SUMMARY): [1-9]", text):
        return "errors reported"
    if step["exit_code"] == 0 and re.search(
        r"ERROR SUMMARY: 0 errors|RACECHECK SUMMARY: 0 hazards", text
    ):
        return "clean"
    return "did not complete"


def classify_run(run_dir):
    """Summarize a validate_gpu.py run directory without reinterpreting it."""
    run_dir = Path(run_dir)
    steps_file = run_dir / "steps.json"
    steps = json.loads(steps_file.read_text()) if steps_file.is_file() else []
    blocking, sanitizer, workflow = [], {}, {}
    valid = {"vector_add": 0, "reduction": 0}
    attempted = {"vector_add": 0, "reduction": 0}
    for s in steps:
        name, base = s["name"], run_dir / s["name"]
        stdout = _read(base, "stdout")
        if name.startswith(BLOCKING_PREFIXES) and s["exit_code"] != 0:
            blocking.append(f"{name}: {s['note'] or 'exit ' + str(s['exit_code'])}")
        if name.startswith("sanitizer/"):
            verdict = sanitizer_verdict(s, stdout + _read(base, "stderr"))
            sanitizer[name.split("/", 1)[1]] = verdict
            if verdict == "errors reported":
                blocking.append(f"{name}: compute-sanitizer reported errors")
        for workload in valid:
            if name.startswith(f"{workload}/"):
                attempted[workload] += 1
                try:
                    doc = json.loads(stdout)
                except json.JSONDecodeError:
                    continue
                device = (doc.get("environment") or {}).get("device") or {}
                if s["exit_code"] == 0 and doc.get("valid") is True and device.get("name"):
                    valid[workload] += 1
        if name.startswith("workflow/"):
            verdicts = re.findall(r"-> (PASS|INCONCLUSIVE|REGRESSION|IMPROVEMENT)", stdout)
            workflow[name.split("/", 1)[1]] = f"exit {s['exit_code']}" + (
                f", {verdicts[-1]}" if verdicts else ""
            )
    return {
        "dir": str(run_dir),
        "steps": len(steps),
        "blocking": blocking,
        "sanitizer": sanitizer,
        "valid": valid,
        "attempted": attempted,
        "workflow": workflow,
    }


def workloads_executed(summary):
    """True only if both workloads produced valid results on a real device."""
    return bool(summary) and all(summary["valid"][w] > 0 for w in ("vector_add", "reduction"))


def archive_name():
    return f"probe-stage5-colab-{gpu_slug(STATUS.get('gpu', {}).get('name'))}-{STARTED}.tar.gz"


def make_archive():
    """Pack the notebook's records and the validation runs, without build trees."""
    archive = ROOT / archive_name()

    def skip(info):
        return None if "/cmake-build" in info.name or info.name.endswith("/cmake-build") else info

    with tarfile.open(archive, "w:gz") as tar:
        tar.add(WORK, arcname="probe-stage5", filter=skip)
        for key in ("quick", "full"):
            run_dir = STATUS.get(key, {}).get("dir")
            if run_dir and Path(run_dir).is_dir():
                tar.add(run_dir, arcname=f"validation-runs/{key}-{Path(run_dir).name}", filter=skip)
    return archive


def offer_download(archive):
    try:
        from google.colab import files
    except ImportError:
        print(f"Not running in Colab; the archive is at {archive}")
        return
    files.download(str(archive))


def summary_lines(complete, reason=None):
    full = STATUS.get("full") or {}
    quick = STATUS.get("quick") or {}
    run = full or quick
    tools = STATUS.get("tools", {})
    gpu = STATUS.get("gpu", {})
    lines = ["STAGE 5 HARDWARE VALIDATION FINISHED" if complete else "STAGE 5 NOT COMPLETE"]
    if not complete:
        lines.append(f"Reason: {reason}")

    def show(value, default="not reached"):
        return default if value in (None, {}, []) else value

    sanitizer = run.get("sanitizer", {})
    if tools.get("compute-sanitizer") == "UNAVAILABLE":
        san = "compute-sanitizer unavailable in this environment"
    elif sanitizer:
        counts = {}
        for verdict in sanitizer.values():
            counts[verdict] = counts.get(verdict, 0) + 1
        san = ", ".join(f"{n} {v}" for v, n in sorted(counts.items()))
    else:
        san = "not reached"
    lines += [
        f"GPU used:              {show(gpu.get('name'))} "
        f"(driver {show(gpu.get('driver_version'))}, "
        f"CUDA {show(gpu.get('driver_cuda_version'))})",
        f"Probe commit tested:   {show(STATUS.get('commit'))}",
        f"Build:                 {show(STATUS.get('build'))}",
        f"C++ tests (ctest):     {show(STATUS.get('ctest'))}",
        f"Python tests (pytest): {show(STATUS.get('pytest'))}",
        f"vector_add:            {_count(run, 'vector_add')}",
        f"reduction:             {_count(run, 'reduction')}",
        f"baseline/compare:      {show(run.get('workflow'))}",
        f"compute-sanitizer:     {san}",
        f"Nsight Systems (nsys): {show(tools.get('nsys'))}",
        f"Nsight Compute (ncu):  {show(tools.get('ncu'))}",
        f"Validation run used:   {'full' if full else ('quick only' if quick else 'none')}",
        f"Archive:               {show(STATUS.get('archive'))}",
    ]
    return lines


def _count(run, workload):
    if not run:
        return "not reached"
    return f"{run['valid'][workload]} of {run['attempted'][workload]} sizes valid on the GPU"


def finish(complete, reason=None):
    """Record the outcome, pack the archive, print the summary, offer the download."""
    STATUS["complete"], STATUS["reason"] = complete, reason
    STATUS["archive"] = archive_name()
    save_status()  # status.json goes into the archive with its final contents
    archive = make_archive()
    print("\n" + "\n".join(summary_lines(complete, reason)))
    offer_download(archive)
    return archive


def stop(reason):
    """End the validation here: pack what was recorded and stop 'Run all'."""
    print("\n" + "=" * 72)
    finish(False, reason)
    print("=" * 72)
    raise StageStopped(reason)


print(f"Recording to {WORK}")

## Stage 0: GPU runtime check

In [ ]:
code, smi = sh("stage0-nvidia-smi", "nvidia-smi")
if code != 0 or "NVIDIA-SMI" not in smi:
    stop(
        "No NVIDIA GPU is attached to this Colab runtime.\n"
        "Go to Runtime -> Change runtime type -> GPU, reconnect, and run again."
    )

code, query = sh(
    "stage0-gpu-query", "nvidia-smi --query-gpu=name,driver_version --format=csv,noheader"
)
rows = [r for r in query.strip().splitlines() if r.strip()]
name, driver = (part.strip() for part in rows[0].split(",")[:2])
cuda = re.search(r"CUDA Version:\s*([\d.]+)", smi)
STATUS["gpu"] = {
    "name": name,
    "driver_version": driver,
    "count": len(rows),
    "driver_cuda_version": cuda.group(1) if cuda else None,
}
STATUS["stages"]["0"] = "ok"
save_status()
print(
    f"\nGPU: {name}\nNVIDIA driver: {driver}\nCUDA version reported by the driver: "
    f"{STATUS['gpu']['driver_cuda_version']}"
)

## Stage 1: Toolchain inspection

Nothing is installed or changed here; this only records what Colab provides. Missing Nsight tools or compute-sanitizer do not stop the validation.

In [ ]:
for name, command in [
    ("stage1-nvidia-smi", "nvidia-smi"),
    ("stage1-nvcc", "nvcc --version"),
    ("stage1-gcc", "gcc --version"),
    ("stage1-gxx", "g++ --version"),
    ("stage1-cmake", "cmake --version"),
    ("stage1-python", f"{sys.executable} --version"),
]:
    sh(name, command)

STATUS["tools"] = {}
for tool in ("nvcc", "cuobjdump", "compute-sanitizer", "nsys", "ncu", "cmake", "gcc", "g++", "git"):
    STATUS["tools"][tool] = "AVAILABLE" if shutil.which(tool) else "UNAVAILABLE"
for tool in ("compute-sanitizer", "nsys", "ncu"):
    if STATUS["tools"][tool] == "AVAILABLE":
        _, out = sh(f"stage1-{tool}-version", f"{tool} --version", quiet=True)
        STATUS["tools"][f"{tool}_version"] = out.strip().splitlines()[-1] if out.strip() else ""
(WORK / "toolchain.json").write_text(json.dumps(STATUS["tools"], indent=2) + "\n")
STATUS["stages"]["1"] = "ok"
save_status()
print()
for tool in ("nvcc", "cuobjdump", "compute-sanitizer", "nsys", "ncu", "cmake", "gcc", "g++", "git"):
    print(f"{tool:<18} {STATUS['tools'][tool]}")

## Stage 2: Upload and clone `probe.bundle`

If `/content/probe.bundle` already exists, for example on a re-run, it is used without asking again.

In [ ]:
if not BUNDLE.is_file():
    try:
        from google.colab import files
    except ImportError:
        stop(f"{BUNDLE} not found, and this is not a Colab runtime to upload it from.")
    print("Select probe.bundle in the upload dialog.")
    uploaded = files.upload()
    bundles = [n for n in uploaded if n.endswith(".bundle")]
    if not bundles:
        stop("No .bundle file was uploaded.")
    BUNDLE.write_bytes(uploaded[bundles[0]])
print(f"{BUNDLE} ({BUNDLE.stat().st_size} bytes)")

shutil.rmtree(REPO, ignore_errors=True)  # always a fresh clone of the bundle
code, _ = sh("stage2-clone", f"git clone {BUNDLE} {REPO}")
if code != 0:
    stop("git could not clone probe.bundle; see the output above.")
sh("stage2-verify", f"git bundle verify {BUNDLE}", cwd=REPO)
sh("stage2-log", "git log -5 --oneline", cwd=REPO)
sh("stage2-status", "git status", cwd=REPO)
_, user = sh("stage2-user-name", "git config user.name", cwd=REPO)
_, email = sh("stage2-user-email", "git config user.email", cwd=REPO)
_, authors = sh("stage2-authors", "git log --format='%an <%ae>' | sort -u", cwd=REPO)
_, commit = sh("stage2-commit", "git rev-parse HEAD", cwd=REPO, quiet=True)
_, fuller = sh("stage2-commit-detail", "git log -1 --format=fuller", cwd=REPO, quiet=True)
STATUS["commit"] = commit.strip()
(WORK / "commit.txt").write_text(fuller)
print(f"\nCommit under test: {STATUS['commit']}")
print(f"Commit authors: {authors.strip()}")
print(
    "git identity in this runtime: "
    f"{user.strip() or '(not set)'} {email.strip() or ''} -- no commits are made here"
)
STATUS["stages"]["2"] = "ok"
save_status()

## Stage 3: Dependency setup

The requirements are read from the repository itself: the Python version and the `dev` extra from `pyproject.toml`, and the CMake minimum from `CMakeLists.txt`. Colab's NVIDIA driver and CUDA toolkit are used as they are; this notebook never installs either. Host tools are installed only if missing or too old, and everything installed is listed.

In [ ]:
import tomllib

project = tomllib.loads((REPO / "pyproject.toml").read_text())["project"]
python_min = version_tuple(project["requires-python"])
if sys.version_info[: len(python_min)] < python_min:
    stop(
        f"Python {sys.version.split()[0]} is older than the required {project['requires-python']}."
    )
if "dev" not in project.get("optional-dependencies", {}):
    stop("pyproject.toml has no 'dev' extra; the repository is not what this notebook expects.")

if shutil.which("nvcc") is None:
    stop("nvcc (the CUDA toolkit) is not available. This notebook does not install CUDA.")
missing = [t for t in ("gcc", "g++", "git") if shutil.which(t) is None]
if missing:
    stop(f"Missing host tools: {', '.join(missing)}.")

installed = []
cmake_min = re.search(
    r"cmake_minimum_required\(VERSION ([\d.]+)\)", (REPO / "CMakeLists.txt").read_text()
).group(1)
cmake_version = ()
if shutil.which("cmake"):
    _, out = sh("stage3-cmake-version", "cmake --version", quiet=True)
    cmake_version = version_tuple(out)
if cmake_version < version_tuple(cmake_min):
    print(f"CMake {cmake_version or 'missing'} is older than {cmake_min}; installing from PyPI.")
    code, _ = sh("stage3-install-cmake", f"{sys.executable} -m pip install 'cmake>={cmake_min}'")
    if code != 0:
        stop(f"Could not install CMake >= {cmake_min}.")
    installed.append(f"cmake>={cmake_min} (pip)")
else:
    print(f"CMake {'.'.join(map(str, cmake_version))} satisfies the minimum {cmake_min}.")

_, before = sh("stage3-pip-before", f"{sys.executable} -m pip freeze", quiet=True)
code, _ = sh("stage3-pip-install", f"{sys.executable} -m pip install -e '.[dev]'", cwd=REPO)
if code != 0:
    stop("pip install -e '.[dev]' failed; see the output above.")
_, after = sh("stage3-pip-after", f"{sys.executable} -m pip freeze", quiet=True)
changes = sorted(set(after.splitlines()) - set(before.splitlines()))
installed += [f"{line} (pip)" for line in changes]
(WORK / "installed.txt").write_text("\n".join(installed) + "\n")
STATUS["installed"] = installed
STATUS["stages"]["3"] = "ok"
save_status()
print("\nInstalled or changed by this notebook:")
print("\n".join(f"  {line}" for line in installed) or "  nothing")

## Stage 4: Clean GPU build and tests

This is a clean Release build with the repository's own architecture policy: native code for 7.5, 8.0, 8.6, 8.9 and 9.0, plus 9.0 PTX. It is not specialized for Colab. Stage 5 checks which of those images matches this GPU. `-DPROBE_REQUIRE_CUDA=ON` makes a missing CUDA compiler an error instead of a silent CPU-only build.

If the product binary cannot be built, the validation stops here with the exact error. Test failures do not stop it; they are recorded.

In [ ]:
shutil.rmtree(REPO / "build", ignore_errors=True)
code, _ = sh(
    "stage4-configure",
    "cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DPROBE_REQUIRE_CUDA=ON",
    cwd=REPO,
)
if code != 0:
    STATUS["build"] = "CMake configuration failed"
    stop("CMake configuration failed. The exact error is above and in logs/stage4-configure.log.")
code, _ = sh("stage4-build", "cmake --build build -j", cwd=REPO)
BINARY = REPO / "build" / "workloads" / "probe-workloads"
if code != 0 or not BINARY.is_file():
    STATUS["build"] = "compilation failed"
    stop(
        "The product binary could not be built. The exact compiler error is above and in "
        "logs/stage4-build.log. Hardware validation cannot continue without it."
    )
STATUS["build"] = "ok (clean Release build)"

code, out = sh("stage4-ctest", "ctest --test-dir build --output-on-failure", cwd=REPO)
summary = re.findall(r"\d+% tests passed.*", out)
STATUS["ctest"] = f"exit {code}" + (f", {summary[-1].strip()}" if summary else "")

code, out = sh("stage4-pytest", f"{sys.executable} -m pytest -rs -p no:cacheprovider", cwd=REPO)
summary = re.findall(r"=+ (.*(?:passed|failed|error).*) =+", out)
STATUS["pytest"] = f"exit {code}" + (f", {summary[-1].strip()}" if summary else "")
STATUS["stages"]["4"] = "ok"
save_status()
print(f"\nctest: {STATUS['ctest']}\npytest: {STATUS['pytest']}")

## Stage 5: Device metadata validation

This compares Probe's own `--device-info` JSON with `nvidia-smi` and `nvcc`. Disagreements are recorded as findings and are not corrected. One gap is expected: Probe records the CUDA version the driver supports, not the NVIDIA driver build number. This stage records whether that gap actually appears.

In [ ]:
code, out = sh("stage5-device-info", f"{BINARY} --device-info")
if code != 0:
    stop(
        f"probe-workloads --device-info exited with {code}: Probe could not use this GPU. "
        "This is a hardware-run finding; the output above is preserved."
    )
info = json.loads(out)["device"]
_, version = sh("stage5-version", f"{BINARY} --version")
_, query = sh(
    "stage5-nvidia-smi-query",
    "nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total "
    "--format=csv,noheader,nounits",
    quiet=True,
)
smi_name, smi_cc, smi_driver, smi_mem = (
    p.strip() for p in query.strip().splitlines()[0].split(",")
)
_, nvcc = sh("stage5-nvcc", "nvcc --version", quiet=True)
toolkit = re.search(r"release ([\d.]+)", nvcc)
toolkit = toolkit.group(1) if toolkit else None

archs = re.search(r"cuda architectures ([^)]+)\)", version)
archs = archs.group(1).split(",") if archs else []
cc = info["compute_capability"].replace(".", "")
native = [a for a in archs if a.split("-")[0] == cc and not a.endswith("-virtual")]
ptx = [a for a in archs if not a.endswith("-real") and int(a.split("-")[0]) <= int(cc)]
if native:
    arch_note = f"native SASS for sm_{cc} is in the build ({native[0]})"
elif ptx:
    arch_note = f"no sm_{cc} SASS; the driver would JIT-compile PTX from {ptx[-1]}"
else:
    arch_note = f"no image for sm_{cc} by exact match; see the validation report"

recorded = json.dumps(info)
checks = [
    ("GPU model", smi_name, info["name"], smi_name == info["name"]),
    (
        "compute capability",
        smi_cc,
        info["compute_capability"],
        smi_cc == info["compute_capability"],
    ),
    ("memory (MiB)", smi_mem, f"{info['global_mem_bytes'] / 2**20:.0f}", None),
    (
        "CUDA version supported by driver",
        STATUS["gpu"]["driver_cuda_version"],
        info["cuda_driver_version"],
        STATUS["gpu"]["driver_cuda_version"] == info["cuda_driver_version"],
    ),
    (
        "CUDA runtime vs toolkit (nvcc)",
        toolkit,
        info["cuda_runtime_version"],
        toolkit == info["cuda_runtime_version"],
    ),
    (
        "NVIDIA driver build",
        smi_driver,
        "present" if smi_driver in recorded else "not recorded",
        smi_driver in recorded,
    ),
]
gap = smi_driver not in recorded
STATUS["device_check"] = {
    "checks": [
        {"field": f, "nvidia_smi_or_nvcc": a, "probe": b, "agrees": ok} for f, a, b, ok in checks
    ],
    "driver_build_gap_present": gap,
    "architectures": archs,
    "architecture_note": arch_note,
}
(WORK / "device-check.json").write_text(json.dumps(STATUS["device_check"], indent=2) + "\n")
STATUS["stages"]["5"] = "ok"
save_status()
print(f"\n{'field':<34} {'system tools':<28} {'Probe':<28} agrees")
for f, a, b, ok in checks:
    print(
        f"{f:<34} {str(a):<28} {str(b):<28} {'compare' if ok is None else ('yes' if ok else 'NO')}"
    )
print(f"\nDriver-build gap present: {'YES' if gap else 'no'}")
print(f"Architecture: {arch_note}")

## Stage 6: Quick validation

`scripts/validate_gpu.py --quick` runs the whole procedure with few sizes and iterations. If any step that exercises Probe fails, or compute-sanitizer reports errors, the notebook stops before the full run and packs the evidence.

In [ ]:
code, out = sh("stage6-quick", f"{sys.executable} scripts/validate_gpu.py --quick", cwd=REPO)
quick_dir = find_run_dir(out)
if quick_dir is None:
    stop("validate_gpu.py --quick did not report where it recorded its results.")
sh(
    "stage6-analyze",
    f"{sys.executable} scripts/analyze_validation.py {quick_dir}",
    cwd=REPO,
    quiet=True,
)
STATUS["quick"] = classify_run(quick_dir)
save_status()
if STATUS["quick"]["blocking"]:
    stop(
        "THE FIRST HARDWARE RUN FOUND A PROBLEM. The quick validation had failing steps:\n  "
        + "\n  ".join(STATUS["quick"]["blocking"])
        + "\nThe full validation was not started. All evidence is in the archive."
    )
if not workloads_executed(STATUS["quick"]):
    stop("The quick validation finished, but no valid CUDA workload result was recorded.")
STATUS["stages"]["6"] = "ok"
save_status()
print(
    f"\nQuick validation: no blocking failures; valid results {STATUS['quick']['valid']}. "
    "Continuing with the full run."
)

## Stage 7: Full Stage 5 validation

`scripts/validate_gpu.py` is the canonical procedure (A–I in `docs/gpu-validation.md`):

- environment capture and a clean build;
- vector_add and reduction correctness at awkward sizes;
- compute-sanitizer;
- timing, warmup, and five independent noise runs per workload;
- the baseline workflow.

Failures here are recorded and shown, not hidden, and the analysis still runs.

In [ ]:
code, out = sh("stage7-full", f"{sys.executable} scripts/validate_gpu.py", cwd=REPO)
full_dir = find_run_dir(out)
if full_dir is None:
    stop("validate_gpu.py did not report where it recorded its results.")
STATUS["full"] = classify_run(full_dir)
STATUS["stages"]["7"] = "ok" if not STATUS["full"]["blocking"] else "completed with failures"
save_status()
if STATUS["full"]["blocking"]:
    print("\nThe full run had failing steps (recorded, not hidden):")
    print("\n".join(f"  {line}" for line in STATUS["full"]["blocking"]))
print(f"\nValid results: {STATUS['full']['valid']} of {STATUS['full']['attempted']} attempted")

## Stage 8: Sanitizer and Nsight availability

compute-sanitizer already ran inside the validation (memcheck, racecheck, synccheck and initcheck on awkward sizes). This stage reports those results. A case counts as **clean** only if it exited 0 *and* the sanitizer printed its own zero-error summary.

Nsight Systems and Nsight Compute are only checked for availability. Integrating them is a later stage.

In [ ]:
tools = STATUS["tools"]
results = STATUS["full"]["sanitizer"]
if tools["compute-sanitizer"] == "UNAVAILABLE":
    print("compute-sanitizer unavailable in this environment")
elif not results:
    print("compute-sanitizer is installed, but no sanitizer step was recorded")
else:
    for case, verdict in results.items():
        print(f"{case:<40} {verdict}")
for tool in ("nsys", "ncu"):
    detail = tools.get(f"{tool}_version", "")
    print(f"{tool:<5} {tools[tool]}{(' (' + detail + ')') if detail else ''}")
STATUS["stages"]["8"] = "ok"
save_status()

## Stage 9: Analysis

`scripts/analyze_validation.py` builds the report from the recorded files alone. It covers the environment comparison, correctness, sanitizer results, reduction error against its bound, timing accounting, warmup, within-run versus between-process noise, and the baseline verdicts. The verdicts are shown exactly as recorded.

In [ ]:
from IPython.display import Markdown, display

code, _ = sh(
    "stage9-analyze",
    f"{sys.executable} scripts/analyze_validation.py {full_dir}",
    cwd=REPO,
    quiet=True,
)
report = Path(full_dir) / "report.md"
if code != 0 or not report.is_file():
    print("The analysis script failed; its output is in logs/stage9-analyze.log.")
else:
    shutil.copy(report, WORK / "report-full.md")
    display(Markdown(report.read_text()))
STATUS["stages"]["9"] = "ok" if code == 0 else f"analysis exit {code}"
save_status()

## Stage 10: Archive and download

One `.tar.gz` holds the notebook logs, the environment and toolchain records, the device check, both validation runs (all raw outputs and result JSON, sanitizer logs, and reports), and the exact commit tested. Build trees and binaries are left out.

In [ ]:
complete = workloads_executed(STATUS["full"])
reason = None if complete else "the full validation recorded no valid result for both workloads"
archive = finish(complete, reason)

## Stage 11: Summary

The summary is printed again here for reference. It is also stored as `status.json` in the archive.

In [ ]:
print("\n".join(summary_lines(STATUS["complete"], STATUS["reason"])))